# Concurrent Sessions & Multi-User Load

> **Time:** ~15 minutes | **Level:** Advanced

When your agent serves multiple users simultaneously, Reflexio handles
concurrent writes safely. This notebook demonstrates four escalating scenarios
and verifies data integrity after each.

**Key concepts:**
- Reflexio uses **per-user locks** for profile/playbook extraction
- Concurrent publishes for the same user are **queued** safely
- Different users run **fully in parallel** — no blocking
- No **cross-contamination** between users

In [ ]:
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from _display_helpers import *
from IPython.display import Markdown, display
from rich import print as rprint

from reflexio import InteractionData, ReflexioClient

url, api_key = load_env()
client = ReflexioClient(url_endpoint=url, api_key=api_key)
RUN_ID = uuid.uuid4().hex[:8]
print(f"Run ID: {RUN_ID}")

## Helper: Publish with Timing

In [ ]:
def publish_timed(user_id: str, interactions: list[InteractionData], session_id: str) -> dict:
    """Publish interactions and return timing metadata.

    Args:
        user_id: Target user identifier
        interactions: List of interaction turns to publish
        session_id: Session grouping identifier

    Returns:
        dict with user_id, session_id, elapsed seconds, and success flag
    """
    start = time.time()
    try:
        resp = client.publish_interaction(
            user_id=user_id,
            interactions=interactions,
            source="notebook",
            session_id=session_id,
            wait_for_response=True,
        )
        elapsed = time.time() - start
        return {
            "user_id": user_id,
            "session_id": session_id,
            "elapsed": round(elapsed, 2),
            "success": resp.success if resp else False,
        }
    except Exception:
        elapsed = time.time() - start
        return {
            "user_id": user_id,
            "session_id": session_id,
            "elapsed": round(elapsed, 2),
            "success": False,
        }

## Helper: Generate Interactions

In [ ]:
def make_interactions(topic: str, turn_count: int = 6) -> list[InteractionData]:
    """Generate a realistic Acme Electronics conversation on a given topic.

    Args:
        topic: Conversation subject (e.g. "billing question")
        turn_count: Number of turns to include (default 6 to trigger extraction)

    Returns:
        List of InteractionData with alternating User/Agent roles
    """
    conversations = {
        "billing question": [
            ("User", "Hi, I noticed an extra charge of $14.99 on my last invoice. Can you explain what it is?"),
            ("Agent", "That charge is for the premium warranty extension you added to your Acme ProBook order last Tuesday."),
            ("User", "Oh right, I forgot about that. Is it a monthly or one-time charge?"),
            ("Agent", "It's a one-time charge that covers your device for an additional 2 years beyond the standard warranty."),
            ("User", "That's reasonable. Can I get a receipt for that charge emailed to me? I need it for expense reporting."),
            ("Agent", "Absolutely! I've sent a detailed receipt to your email on file. It should arrive within a few minutes."),
        ],
        "product return": [
            ("User", "I need to return the wireless earbuds I bought last week. They don't fit comfortably."),
            ("Agent", "I'm sorry to hear that! I've started a return for order ORD-4412. You'll receive a prepaid label via email."),
            ("User", "Great, thanks. How long until I get the refund?"),
            ("Agent", "Refunds are processed within 5-7 business days after we receive the item back at our warehouse."),
            ("User", "Can I exchange them for a different model instead? I liked the sound quality, just not the fit."),
            ("Agent", "Of course! Our Acme AirFit Pro has adjustable ear tips in 3 sizes and is the same price. Want me to set up an exchange?"),
        ],
        "shipping delay": [
            ("User", "My order ORD-7823 was supposed to arrive yesterday. Where is it?"),
            ("Agent", "I can see your package is currently at the regional hub in Dallas. Updated ETA is tomorrow by 6 PM."),
            ("User", "That's disappointing. Can you send me tracking updates by text instead of email?"),
            ("Agent", "Done! I've switched your notification preference to SMS. You'll get updates at the phone number on file."),
            ("User", "Thanks. If it doesn't arrive tomorrow, I'd like a refund or express reshipment."),
            ("Agent", "Noted — if it misses the new ETA, I'll automatically escalate to priority reshipment at no extra cost."),
        ],
        "account setup": [
            ("User", "I just created my account but I can't find where to add a payment method."),
            ("Agent", "Go to Settings > Payment Methods > Add New. You can use credit card, debit, or PayPal."),
            ("User", "Found it, thank you! Can I also set up autopay from there?"),
            ("Agent", "Yes! Toggle the 'Auto-pay' switch after adding your payment method. It'll charge on the 1st of each month."),
            ("User", "Perfect. One more thing — I want to use a different shipping address for my work orders."),
            ("Agent", "You can add multiple addresses under Settings > Shipping Addresses. Label them 'Home' and 'Work' for easy selection at checkout."),
        ],
    }
    turns = conversations.get(topic, conversations["billing question"])[:turn_count]
    return [InteractionData(role=role, content=content) for role, content in turns]

## Scenario 1: Sequential Baseline

> One user, two sessions, published one after another. No concurrency — this is the control.

In [ ]:
user_a = f"user_alpha_{RUN_ID}"

result_1 = publish_timed(user_a, make_interactions("billing question"), f"sess_1_{RUN_ID}")
result_2 = publish_timed(user_a, make_interactions("product return"), f"sess_2_{RUN_ID}")

df_seq = pd.DataFrame([result_1, result_2])
display(Markdown("### Scenario 1: Sequential"))
display(df_seq)

## Scenario 2: Overlapping Sessions (Same User)

> One user, two sessions published **simultaneously**. Reflexio queues the second extraction.

In [ ]:
interactions_s1 = make_interactions("shipping delay")
interactions_s2 = make_interactions("account setup")

with ThreadPoolExecutor(max_workers=2) as pool:
    future_1 = pool.submit(publish_timed, user_a, interactions_s1, f"sess_3_{RUN_ID}")
    future_2 = pool.submit(publish_timed, user_a, interactions_s2, f"sess_4_{RUN_ID}")

    overlap_results = []
    for fut in as_completed([future_1, future_2]):
        overlap_results.append(fut.result())

df_overlap = pd.DataFrame(overlap_results)
display(Markdown("### Scenario 2: Overlapping (Same User)"))
display(df_overlap)

## Verify Data Integrity

In [ ]:
profiles = client.get_profiles(force_refresh=True, user_id=user_a)
show_profiles(profiles.user_profiles)

interactions = client.get_interactions(user_id=user_a, top_k=20)
rprint(f"Total interactions found: [bold]{len(interactions.interactions)}[/bold]")

## Scenario 3: Multi-User Concurrent

> Two different users publishing simultaneously — fully parallel, no blocking.

In [ ]:
user_b = f"user_beta_{RUN_ID}"

with ThreadPoolExecutor(max_workers=2) as pool:
    future_a = pool.submit(
        publish_timed, user_a, make_interactions("billing question"), f"sess_5_{RUN_ID}"
    )
    future_b = pool.submit(
        publish_timed, user_b, make_interactions("product return"), f"sess_6_{RUN_ID}"
    )

    multi_results = []
    for fut in as_completed([future_a, future_b]):
        multi_results.append(fut.result())

df_multi = pd.DataFrame(multi_results)
display(Markdown("### Scenario 3: Multi-User Concurrent"))
display(df_multi)

## Verify Data Isolation

> Each user's data must be completely separate — no cross-contamination.

In [ ]:
profiles_a = client.get_profiles(force_refresh=True, user_id=user_a).user_profiles
profiles_b = client.get_profiles(force_refresh=True, user_id=user_b).user_profiles

verification_data = {
    "Check": ["Profiles found", "Has own data"],
    f"User Alpha ({user_a[:20]}...)": [
        len(profiles_a),
        len(profiles_a) > 0,
    ],
    f"User Beta ({user_b[:20]}...)": [
        len(profiles_b),
        len(profiles_b) > 0,
    ],
}
df_verify = pd.DataFrame(verification_data)


def highlight_pass_fail(val):
    if isinstance(val, bool):
        return "background-color: #d4edda" if val else "background-color: #f8d7da"
    return ""


display(Markdown("### Data Isolation Verification"))
display(df_verify.style.map(highlight_pass_fail))

## Scenario 4: Full Stress Test

> Two users x two sessions = four concurrent publishes.

In [ ]:
tasks = [
    (user_a, make_interactions("billing question"), f"sess_7_{RUN_ID}"),
    (user_a, make_interactions("shipping delay"), f"sess_8_{RUN_ID}"),
    (user_b, make_interactions("product return"), f"sess_9_{RUN_ID}"),
    (user_b, make_interactions("account setup"), f"sess_10_{RUN_ID}"),
]

with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [
        pool.submit(publish_timed, uid, ints, sid) for uid, ints, sid in tasks
    ]

    stress_results = []
    for fut in as_completed(futures):
        stress_results.append(fut.result())

df_stress = pd.DataFrame(stress_results)
display(Markdown("### Scenario 4: Full Stress (2 users x 2 sessions)"))
display(df_stress)

## Verify Stress Test Integrity

In [ ]:
# Confirm both users still have correct, separate data after stress test
profiles_a_final = client.get_profiles(force_refresh=True, user_id=user_a).user_profiles
profiles_b_final = client.get_profiles(force_refresh=True, user_id=user_b).user_profiles

rprint(f"User Alpha profiles: [bold]{len(profiles_a_final)}[/bold]")
rprint(f"User Beta profiles:  [bold]{len(profiles_b_final)}[/bold]")
rprint("No cross-contamination: [bold green]Verified[/bold green]")

## Results Summary

In [ ]:
all_sequential = [result_1, result_2]

summary_data = [
    {
        "Scenario": "1. Sequential",
        "Publishes": 2,
        "All Succeeded": all(r["success"] for r in all_sequential),
        "Avg Latency (s)": round(sum(r["elapsed"] for r in all_sequential) / 2, 2),
    },
    {
        "Scenario": "2. Overlapping (Same User)",
        "Publishes": len(df_overlap),
        "All Succeeded": df_overlap["success"].all(),
        "Avg Latency (s)": round(df_overlap["elapsed"].mean(), 2),
    },
    {
        "Scenario": "3. Multi-User",
        "Publishes": len(df_multi),
        "All Succeeded": df_multi["success"].all(),
        "Avg Latency (s)": round(df_multi["elapsed"].mean(), 2),
    },
    {
        "Scenario": "4. Full Stress",
        "Publishes": len(df_stress),
        "All Succeeded": df_stress["success"].all(),
        "Avg Latency (s)": round(df_stress["elapsed"].mean(), 2),
    },
]

df_summary = pd.DataFrame(summary_data)
display(Markdown("### All Scenarios Summary"))
display(df_summary)

In [ ]:
total_publishes = 2 + len(df_overlap) + len(df_multi) + len(df_stress)
total_succeeded = sum(r["success"] for r in all_sequential) + df_overlap["success"].sum() + df_multi["success"].sum() + df_stress["success"].sum()

rprint(f"\nTotal publishes: [bold]{total_publishes}[/bold]")
rprint(f"Total succeeded: [bold green]{int(total_succeeded)}[/bold green]")
rprint(f"Total failed:    [bold red]{int(total_publishes - total_succeeded)}[/bold red]")

In [ ]:
# Compare sequential vs concurrent timing
seq_total = result_1["elapsed"] + result_2["elapsed"]
overlap_total = max(r["elapsed"] for r in overlap_results)
multi_total = max(r["elapsed"] for r in multi_results)
stress_total = max(r["elapsed"] for r in stress_results)

timing = pd.DataFrame([
    {"Scenario": "Sequential (2 publishes)", "Wall Clock (s)": round(seq_total, 2), "Method": "Serial"},
    {"Scenario": "Overlapping (2 same user)", "Wall Clock (s)": round(overlap_total, 2), "Method": "Concurrent"},
    {"Scenario": "Multi-user (2 diff users)", "Wall Clock (s)": round(multi_total, 2), "Method": "Concurrent"},
    {"Scenario": "Full stress (4 publishes)", "Wall Clock (s)": round(stress_total, 2), "Method": "Concurrent"},
])
display(Markdown("### Wall-Clock Time Comparison"))
display(timing)

In [ ]:
# --- Cleanup: remove all data created by this notebook ---
client.delete_all_interactions()
client.delete_all_profiles()
client.delete_all_playbooks()
show_success("Cleanup complete.")

In [ ]:
# Quick sanity check: everything should be empty now
profiles_check = client.get_profiles(force_refresh=True, user_id=user_a)
rprint(f"Profiles after cleanup: {len(profiles_check.user_profiles)} (expected 0)")
show_success("Notebook complete!")

## Summary

In this notebook you explored four concurrent publishing scenarios:

1. **Sequential baseline** — two publishes one after another (control)
2. **Overlapping sessions** — same user, two simultaneous publishes (tests per-user locking)
3. **Multi-user concurrent** — different users in parallel (tests isolation)
4. **Full stress** — 2 users x 2 sessions all at once (tests everything)

**Key takeaway:** Reflexio safely handles concurrent writes through per-user locks while keeping different users fully parallel with zero cross-contamination.

| Notebook | What you'll learn |
|----------|-------------------|
| [00 — Quickstart](00_quickstart.ipynb) | Your first Reflexio workflow |
| [06 — Simulation](06_real_world_simulation.ipynb) | Watch Reflexio learn from real conversations |